# Практическая работа № 3 (пример выполнения)
## Рефакторинг «божественного объекта» по принципам SOLID

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 3 — Принципы SOLID и проектирование классов  
**Проект:** «Тайна исчезнувшего дива» — мистический детектив по мотивам цикла Виктора Дашкевича о графе Аверине  
**Формат:** демонстрационный блокнот с полным разбором задания  
**Цель:** показать, как разложить монолитный класс `Game` на систему взаимодействующих объектов, применяя SRP, OCP, LSP, ISP, DIP

---

## 📖 Контекст: где мы находимся

После заданий №1 и №2 проект «Тайна исчезнувшего дива» имеет:

| Слой | Классы |
|---|---|
| **Абстракции** | `GameObject` (ABC), `Movable`/`Drawable`/`Collidable` (Protocol) |
| **Сущности** | `Detective`, `Div`, `Location`, `Clue` |
| **Конфиги** | `GameConfig`, `DetectiveConfig`, `DivConfig`, `LocationConfig`, `ClueData` |
| **Сквозное** | `LoggingMixin`, `UI`/`FakeUI` |
| **Оркестрация** | `Game` |

## 🚧 Какие проблемы остались

Класс `Game` — классический **God Object**. Он делает всё сразу:

```python
class Game:
    def run(self):
        # 1. Выводит заголовок (Renderer)
        # 2. Читает ввод игрока (InputHandler)
        # 3. Управляет локацией (State)
        # 4. Обрабатывает осмотр (InvestigationSystem)
        # 5. Обрабатывает диалог (DialogueSystem)
        # 6. Обрабатывает перемещение (MovementSystem)
        # 7. Проверяет победу (VictoryChecker)
```

**Четыре конкретные проблемы:**

1. **Нарушение SRP.** Семь ответственностей в одном классе. Изменение любой из них затрагивает `Game`.
2. **Нарушение OCP.** Добавление новой команды (например, «применить Плеть») требует правки `Game.run()`.
3. **Нарушение DIP.** `Game` создаёт `Detective`, `Div`, `Location` внутри себя. Заменить их на mock для теста невозможно.
4. **Нарушение ISP.** Методы `Game` не разделены по интерфейсам — клиент получает доступ ко всему сразу.

## 🎯 Что демонстрирует этот блокнот и как это улучшит проект

| Инструмент | Проблема | Улучшение |
|---|---|---|
| **`Action` (dataclass)** | Команды передаются как строки | Типизированный объект действия |
| **`GameState`** | Состояние размазано по `Game` | Всё состояние в одном объекте |
| **`GameSystem` (Protocol)** | Системы не имеют общего контракта | Единый интерфейс для всех обработчиков |
| **`InputHandler`** | `Game` читает ввод | Изолированный ввод, легко подменить |
| **`Renderer`** | `Game` печатает | Изолированный вывод, легко заменить на GUI |
| **`InvestigationSystem`** | Логика осмотра в `Game` | Система с одной ответственностью |
| **`DialogueSystem`** | Логика диалога в `Game` | Система с одной ответственностью |
| **`MovementSystem`** | Логика перемещения в `Game` | Система с одной ответственностью |
| **`VictoryChecker`** | Проверка победы в `Game` | Изолированная политика победы |
| **Новый `Game`** | God Object | Оркестратор систем через абстракции |

В конце блокнота проект будет готов к паттернам проектирования (модули 4–7).

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# Теория: принципы SOLID

## Что такое SOLID

**SOLID** — акроним пяти принципов объектно-ориентированного проектирования, сформулированных Робертом Мартином. Они описывают, как проектировать классы, чтобы код был расширяемым, тестируемым и сопровождаемым.

### S — Single Responsibility Principle (SRP)

**У класса должна быть только одна причина для изменения.** Если класс делают ввод, вывод и логику — это три причины для изменения.

**Пример нарушения:** `Game` знает о вводе, выводе, локациях и победе. Изменение UI ломает игровой цикл; изменение правил победы — тоже.

### O — Open/Closed Principle (OCP)

**Классы должны быть открыты для расширения, но закрыты для изменения.** Добавление нового поведения не должно требовать правки существующих классов.

**Пример нарушения:** добавление команды «Плеть» требует правки `if/elif` в `Game.run()`.

### L — Liskov Substitution Principle (LSP)

**Объекты подкласса должны заменять объекты родителя без изменения поведения.** Если `Square` наследуется от `Rectangle`, но ломает инварианты — LSP нарушен.

**Пример нарушения:** если `FakeUI` наследует `UI`, но не может обрабатывать `get_input()`, его нельзя подставлять вместо `UI`.

### I — Interface Segregation Principle (ISP)

**Клиент не должен зависеть от методов, которые он не использует.** Много маленьких интерфейсов лучше одного большого.

**Пример нарушения:** если `GameSystem` требует `handle()`, `validate()`, `rollback()`, `describe()`, а система использует только `handle()` — ISP нарушен.

### D — Dependency Inversion Principle (DIP)

**Модули высокого уровня не должны зависеть от модулей низкого уровня. Оба должны зависеть от абстракций.**

**Пример нарушения:** `Game` создаёт `Detective` и `Location` напрямую. Заменить их на mock или другую реализацию невозможно.

## Как это улучшит проект

Применение SOLID даст:
- **Тестируемость.** Каждая система тестируется изолированно.
- **Расширяемость.** Новая команда = новая система, без правки `Game`.
- **Читаемость.** Каждый класс делает одну вещь.
- **Гибкость.** UI можно заменить на GUI, не меняя логику.

## Для углублённого изучения

- Martin R. C. «Clean Architecture», часть III («Принципы SOLID»).
- Martin R. C. «Clean Code», главы о классах.
- [Refactoring Guru: SOLID Principles](https://refactoring.guru/design-patterns/solid)
- [Real Python: SOLID Principles in Python](https://realpython.com/solid-principles-python/)
- [Python docs: typing.Protocol](https://typing.python.org/en/latest/spec/protocol.html)

---
# Шаг 0. Подготовка

## Зачем этот шаг

Мы продолжаем в том же репозитории. Нужно:
- убедиться, что окружение активно;
- создать ветку для работы над рефакторингом;
- зафиксировать текущее состояние как «до».

## Команды в терминале

```bash
git checkout main
git checkout -b module3-solid
pytest -v  # убедиться, что всё зелёное
```

**Зачем отдельная ветка:** рефакторинг — рискованное дело. Если что-то пойдёт не так, можно вернуться к `main`.

---
# Шаг 1. Аудит: находим нарушения SOLID

## Зачем этот шаг

Прежде чем рефакторить, нужно **точно понять, что именно сломано**. Преждевременный рефакторинг без аудита приводит к хаосу.

## Что мы ищем

Ищите в коде пять признаков:

1. **Божественный объект (God Object).** Класс, который делает слишком много.
2. **Анемичная модель (Anemic Domain Model).** Класс только с данными, без поведения.
3. **Длинные методы с if/elif.** Признак нарушения OCP.
4. **Прямое создание зависимостей.** Признак нарушения DIP.
5. **Методы, которые не используются всеми клиентами.** Признак нарушения ISP.

## Текущий `Game` (до рефакторинга)

Ниже — версия `Game` после заданий №1 и №2. Прочитайте её и отметьте нарушения:

In [ ]:
%%writefile mystic_detective/game/game_before.py
"""Версия Game ДО рефакторинга — God Object.

Демонстрирует нарушения SOLID.
Только для сравнения — не использовать в проде.
"""

import random
from entities.detective_v2 import Detective
from entities.div import Div
from entities.location import Location
from entities.clue import Clue
from core.config import DetectiveConfig, GameConfig
from ui.ui import UI


class GameGodObject:
    """God Object: делает всё сразу.

    Нарушения:
    - SRP: ввод, вывод, состояние, логика, победа — в одном классе.
    - OCP: добавление команды требует правки run().
    - DIP: создаёт Detective, Div, Location внутри себя.
    - ISP: клиент получает весь класс целиком.
    """

    def __init__(self, ui: UI | None = None):
        # DIP-нарушение: создаём зависимости напрямую
        self._ui = ui if ui is not None else UI()
        self._div = Div("Кузя", form="кот", loyalty=50)
        self._detective = Detective(
            config=DetectiveConfig(),
            game_config=GameConfig(),
            div=self._div,
        )
        self._locations = self._build_locations()
        self._current_location = self._locations['особняк']

    def _build_locations(self) -> dict:
        """SRP-нарушение: создание локаций — отдельная ответственность."""
        loc1 = Location("особняк", "Старый особняк Авериных.")
        loc1.add_clue(Clue("Странный запах гари", is_key=False))
        loc1.add_character("Дворецкий")
        loc2 = Location("академия", "Академия колдовских наук.")
        loc2.add_clue(Clue("Записка с планом Пустоши", is_key=True))
        loc2.add_character("Ректор")
        loc3 = Location("пустошь", "Ледяная пустыня.")
        loc3.add_clue(Clue("Следы Владимира", is_key=True))
        loc3.add_character("Древний див")
        return {"особняк": loc1, "академия": loc2, "пустошь": loc3}

    def run(self) -> None:
        """SRP-нарушение: цикл делает всё.

        OCP-нарушение: if/elif по командам.
        """
        # Renderer-нарушение
        self._ui.show_message("=== ТАЙНА ИСЧЕЗНУВШЕГО ДИВА ===")
        while self._detective.energy > 0:
            # Renderer
            self._ui.show_message(f"\n=== {self._current_location.name.upper()} ===")
            self._ui.show_message(self._current_location.description)
            self._ui.show_message(f"{self._detective}")
            self._ui.show_message("\n1. Осмотреть\n2. Поговорить\n3. Переместиться\n4. Выйти")
            # InputHandler
            choice = self._ui.get_input("Выбор: ")
            # OCP-нарушение: каждая новая команда = новый elif
            if choice == '1':
                self._ui.show_message(self._detective.investigate(self._current_location.name))
                clue = self._current_location.search()
                if clue:
                    self._detective.add_clue(clue)
                    self._ui.show_message(f"Найдена улика: {clue}")
                else:
                    self._ui.show_message("Здесь больше ничего нет.")
            elif choice == '2':
                character = self._current_location.get_character()
                if character:
                    self._ui.show_message(f"{character} рассказывает что-то важное...")
                else:
                    self._ui.show_message("Здесь не с кем говорить.")
            elif choice == '3':
                names = "/".join(self._locations.keys())
                loc_name = self._ui.get_input(f"Куда? ({names}): ")
                if loc_name in self._locations:
                    self._current_location = self._locations[loc_name]
                    self._detective.rest(5)
                else:
                    self._ui.show_message("Такой локации нет.")
            elif choice == '4':
                break
            # VictoryChecker
            key_clues = [c for c in self._detective.clues if c.is_key]
            if len(key_clues) >= 2:
                self._ui.show_message("\nВы раскрыли тайну!")
                return
        self._ui.show_message("\nЭнергия закончилась.")


## Таблица нарушений

| Принцип | Где нарушение | Как проявляется |
|---|---|---|
| **SRP** | `GameGodObject` | 7 ответственностей в одном классе |
| **SRP** | `_build_locations()` | Создание данных внутри класса |
| **OCP** | `run()` | `if/elif` по командам |
| **DIP** | `__init__` | Создаёт `Detective`, `Div`, `Location` напрямую |
| **ISP** | Весь класс | Нет разделения на интерфейсы |
| **LSP** | — | Не нарушен (но нужно следить при рефакторинге) |

**Вывод:** нужен рефакторинг по всем пяти принципам. Приступаем.

---
# Шаг 2. Абстракции: `Action` и `GameSystem`

## Зачем этот шаг

Прежде чем разносить логику по системам, нужно определить **общий язык**: как команды передаются и как системы их обрабатывают.

**Проблема сейчас:** команды — это строки `'1'`, `'2'`, `'3'`. Их смысл знает только `Game.run()`. Новая система не может «подписаться» на команду.

**Решение:** ввести `Action` — типизированный объект команды, и `GameSystem` — протокол для обработчиков.

## Теория: зачем абстракции

### `Action` как dataclass

`Action` — иммутабельный объект с именем команды и аргументами. Использование `frozen=True` даёт:
- невозможность случайно изменить команду в середине обработки;
- хешируемость (можно использовать в set/dict);
- автоматический `__eq__` для тестов.

### `GameSystem` как Protocol

`GameSystem` определяет контракт: «система умеет обрабатывать команду». Используется **Protocol**, а не ABC, потому что:
- системы могут быть из разных источников (плагины, пользовательские);
- важно «что умеет», а не «чем является»;
- не хотим заставлять систему наследоваться от нашего класса.

**Метод `can_handle` + `handle`** — это классический паттерн **Chain of Responsibility** (цепочка обязанностей), который мы изучим позже. Пока просто используем структуру: система сама говорит, обрабатывает ли она команду.

## Как это улучшит проект

**До:**
```python
if choice == '1':
    ...
elif choice == '2':
    ...
```

**После:**
```python
for system in self._systems:
    if system.can_handle(action):
        system.handle(action, state)
        break
```

Добавление новой команды = новый класс-система, без правки `Game`.

## Для углублённого изучения

- [Refactoring Guru: Chain of Responsibility](https://refactoring.guru/design-patterns/chain-of-responsibility)
- [Real Python: Protocols](https://realpython.com/python-protocol/)
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)

In [ ]:
%%writefile mystic_detective/core/action.py
"""Action — типизированный объект команды игрока.

Заменяет строки '1', '2', '3' на объекты с именем и аргументами.
Иммутабельный (frozen=True) — команду нельзя изменить в середине обработки.
"""

from dataclasses import dataclass


@dataclass(frozen=True)
class Action:
    """Команда игрока: имя и аргументы.

    Примеры:
        Action("investigate")
        Action("move", ("академия",))
        Action("talk", ("Ректор",))
    """

    name: str
    args: tuple = ()

    def __str__(self) -> str:
        if self.args:
            return f"{self.name}({', '.join(map(str, self.args))})"
        return self.name


In [ ]:
%%writefile mystic_detective/core/game_system.py
"""GameSystem — протокол для всех игровых систем.

Каждая система:
- имеет имя (совпадает с именем Action, который обрабатывает);
- умеет сказать, обрабатывает ли она данное действие (can_handle);
  в простейшем случае — сравнение имён;
- умеет обработать действие (handle), изменяя состояние игры.

Это позволяет Game работать с системами единообразно.
"""

from typing import Protocol, runtime_checkable

from core.action import Action


@runtime_checkable
class GameSystem(Protocol):
    """Контракт для обработчика команд игрока."""

    @property
    def name(self) -> str:
        """Имя системы (совпадает с именем Action)."""
        ...

    def can_handle(self, action: Action) -> bool:
        """Вернуть True, если система обрабатывает это действие."""
        ...

    def handle(self, action: Action, state: "GameState", ui: "UI") -> None:
        """Обработать действие, изменив состояние игры."""
        ...


---
# Шаг 3. `GameState` — состояние игры

## Зачем этот шаг

**Проблема:** сейчас состояние размазано по `Game`:
- `self._detective`
- `self._locations`
- `self._current_location`

Чтобы системе обработать команду, ей приходится получать весь `Game`. Это нарушает SRP: системы зависят от `Game`, хотя нуждаются только в состоянии.

**Решение:** вынести состояние в отдельный класс `GameState`. Системы получают `GameState`, а не `Game`.

## Теория: почему состояние — отдельный класс

**Принцип SRP:** хранение состояния и его изменение — разные ответственности. `GameState` **хранит** состояние. Системы **изменяют** его.

**Что хранит `GameState`:**
- детектив (с его энергией, уликами);
- локации;
- текущую локацию;
- счётчик ходов.

**Что НЕ хранит `GameState`:**
- системы (они не состояние, они поведение);
- UI (он не состояние, он инфраструктура);
- рендерер (он читает состояние, но не является им).

**Метод `move_to()`** реализует проверку и переход. Он не печатает, не логирует — только меняет состояние. Печать — ответственность системы или рендерера.

## Как это улучшит проект

**До:**
```python
system.handle(action, game)  # система получает весь Game
```

**После:**
```python
system.handle(action, state, ui)  # система получает только состояние и UI
```

Теперь систему можно тестировать с mock-состоянием, не создавая весь `Game`.

## Для углублённого изучения

- [Refactoring Guru: Extract Class](https://refactoring.guru/extract-class)
- Martin Fowler. «Refactoring», глава о Extract Class.

In [ ]:
%%writefile mystic_detective/core/game_state.py
"""GameState — всё состояние игры в одном объекте.

Отделяет состояние от логики:
- GameState хранит данные;
- системы изменяют их через методы GameState;
- рендерер читает их через свойства.

Это делает системы тестируемыми с mock-состоянием.
"""

from entities.detective_v2 import Detective
from entities.location import Location


class GameState:
    """Состояние игры: детектив, локации, текущая позиция, ходы."""

    def __init__(
        self,
        detective: Detective,
        locations: dict[str, Location],
        start_location: str,
    ):
        if start_location not in locations:
            raise ValueError(f"Стартовая локация {start_location!r} не найдена")
        self._detective = detective
        self._locations = locations
        self._current_location_name = start_location
        self._turn = 0

    @property
    def detective(self) -> Detective:
        return self._detective

    @property
    def locations(self) -> dict[str, Location]:
        return self._locations

    @property
    def current_location(self) -> Location:
        return self._locations[self._current_location_name]

    @property
    def current_location_name(self) -> str:
        return self._current_location_name

    @property
    def turn(self) -> int:
        return self._turn

    def move_to(self, location_name: str) -> bool:
        """Переместиться в локацию. Возвращает True при успехе."""
        if location_name not in self._locations:
            return False
        self._current_location_name = location_name
        self._turn += 1
        return True

    def tick(self) -> None:
        """Увеличить счётчик ходов."""
        self._turn += 1


---
# Шаг 4. `InputHandler` — ввод игрока

## Зачем этот шаг

**Проблема:** `Game.run()` напрямую вызывает `self._ui.get_input()` и разбирает строки `'1'`, `'2'`, `'3'`. Это:
- смешивает ввод с игровым циклом;
- не тестируется (нельзя проверить логику ввода без UI);
- не расширяется (новая команда = правка `run`).

**Решение:** вынести ввод в `InputHandler`, который:
- читает строку от UI;
- преобразует её в объект `Action`;
- возвращает `None` при неизвестной команде.

## Теория: SRP на практике

**Одна причина для изменения.** `InputHandler` меняется только если:
- меняется формат команд (например, вместо цифр — буквы);
- добавляется новая команда;
- меняется способ ввода (клавиатура, геймпад, голос).

Логика обработки команд (`investigate`, `talk`, `move`) — **не** ответственность `InputHandler`. Он только переводит ввод в `Action`.

## Как это улучшит проект

**До:**
```python
choice = self._ui.get_input("Выбор: ")
if choice == '1': ...
elif choice == '2': ...
```

**После:**
```python
action = self._input.read_action()
if action is None:
    ui.show_message("Неизвестная команда")
else:
    self._dispatch(action)
```

Завтра можно добавить команду `'5'` — правится только словарь `COMMANDS`.

## Для углублённого изучения

- Martin R. C. «Clean Code», глава 3 (функции) и глава 10 (классы).
- [Refactoring Guru: Extract Class](https://refactoring.guru/extract-class)

In [ ]:
%%writefile mystic_detective/game/input_handler.py
"""InputHandler — превращает ввод игрока в Action.

Ответственность (SRP):
- читать строку от UI;
- преобразовывать её в объект Action;
- возвращать None при неизвестной команде.

НЕ отвечает за:
- обработку команд (это системы);
- вывод (это Renderer);
- изменение состояния (это GameState).
"""

from core.action import Action
from ui.ui import UI


class InputHandler:
    """Читает команды игрока и превращает их в Action."""

    COMMANDS = {
        '1': 'investigate',
        '2': 'talk',
        '3': 'move',
        '4': 'quit',
    }

    def __init__(self, ui: UI):
        self._ui = ui

    def read_action(self) -> Action | None:
        """Прочитать команду и вернуть Action или None."""
        raw = self._ui.get_input("Выбор: ").strip()
        if raw not in self.COMMANDS:
            return None
        name = self.COMMANDS[raw]
        if name == 'move':
            target = self._ui.get_input("Куда: ").strip()
            return Action(name, (target,))
        return Action(name)

    def read_location_choice(self, available: list[str]) -> str | None:
        """Прочитать название локации и проверить, что она есть в списке."""
        prompt = f"Куда? ({'/'.join(available)}): "
        choice = self._ui.get_input(prompt).strip()
        return choice if choice in available else None


---
# Шаг 5. `Renderer` — вывод

## Зачем этот шаг

**Проблема:** `Game.run()` сам печатает заголовок, статус, меню. Это:
- смешивает вывод с логикой;
- не позволяет заменить консольный вывод на GUI;
- не тестируется (что именно выводится — проверяется вручную).

**Решение:** вынести весь вывод в `Renderer`.

## Теория: SRP и замена представления

**Renderer** читает состояние и превращает его в вывод. Он **не изменяет** состояние. Это делает его чистым «представлением».

**Зачем это в игре:** сегодня вывод — консольный. Завтра — Pygame. Послезавтра — веб. Если логика вывода изолирована, замена UI не трогает игровую логику.

**Методы Renderer:**
- `render(state)` — отрисовать текущее состояние;
- `show_menu()` — показать меню команд;
- `show_message(text)` — вывести произвольное сообщение.

**Что важно:** `Renderer` не знает о системах, `Action` или логике. Он знает только о `GameState` и `UI`.

## Как это улучшит проект

**До:**
```python
self._ui.show_message(f"=== {self._current_location.name.upper()} ===")
self._ui.show_message(self._current_location.description)
```

**После:**
```python
self._renderer.render(state)
```

Логика вывода — в одном месте. Легко поменять формат.

## Для углублённого изучения

- [Refactoring Guru: Extract Class](https://refactoring.guru/extract-class)
- Martin R. C. «Clean Architecture», глава о разделении слоёв.

In [ ]:
%%writefile mystic_detective/game/renderer.py
"""Renderer — вывод состояния игры на экран.

Ответственность (SRP):
- превращать GameState в текст и отправлять в UI.

НЕ отвечает за:
- изменение состояния;
- обработку команд;
- проверку победы.

Завтра легко заменить на PygameRenderer или WebRenderer.
"""

from core.game_state import GameState
from ui.ui import UI


class Renderer:
    """Отрисовывает состояние игры в UI."""

    MENU = (
        "\n1. Осмотреть место\n"
        "2. Поговорить\n"
        "3. Переместиться\n"
        "4. Выйти"
    )

    def __init__(self, ui: UI):
        self._ui = ui

    def show_title(self, title: str) -> None:
        self._ui.show_message(f"=== {title} ===")

    def render(self, state: GameState) -> None:
        """Отрисовать текущее состояние."""
        loc = state.current_location
        self._ui.show_message(f"\n=== {loc.name.upper()} ===")
        self._ui.show_message(loc.description)
        self._ui.show_message(f"{state.detective}")

    def show_menu(self) -> None:
        self._ui.show_message(self.MENU)

    def show_message(self, text: str) -> None:
        self._ui.show_message(text)


---
# Шаг 6. Системы: `InvestigationSystem`, `DialogueSystem`, `MovementSystem`

## Зачем этот шаг

**Проблема:** логика осмотра, диалога и перемещения размазана по `Game.run()` в виде `if/elif`-блоков. Это:
- нарушает OCP (новая команда = правка `run`);
- нарушает SRP (логика разных команд в одном месте);
- не тестируется (нельзя вызвать «обработать осмотр» без создания всего `Game`).

**Решение:** каждая команда — отдельная система (`GameSystem`).

## Теория: системы как единицы поведения

Каждая система:
- имеет **имя**, совпадающее с `Action.name`;
- реализует `can_handle(action)` — обычно `action.name == self.name`;
- реализует `handle(action, state, ui)` — изменяет состояние и выводит сообщения.

### `InvestigationSystem`

Обрабатывает `Action("investigate")`:
1. Вызывает `state.detective.investigate(location)` — тратит энергию, логирует.
2. Пытается найти улику через `location.search()`.
3. Если улика есть — добавляет её в детектива, выводит сообщение.
4. Если нет — выводит «здесь больше ничего нет».

### `DialogueSystem`

Обрабатывает `Action("talk")`:
1. Получает персонажа через `location.get_character()`.
2. Если персонаж есть — выводит реплику.
3. Если нет — сообщает, что говорить не с кем.

### `MovementSystem`

Обрабатывает `Action("move", (target,))`:
1. Проверяет, что `target` есть в `state.locations`.
2. Вызывает `state.move_to(target)` — меняет текущую локацию.
3. Восстанавливает немного энергии через `detective.rest()` — перемещение утомляет.

## Как это улучшит проект

**До:** один класс `Game` знает про все команды.

**После:** каждая команда — независимая система.
- Добавить новую команду = новый класс, **без правки `Game`** (OCP).
- Тестировать систему = создать `GameState` и `FakeUI`, вызвать `handle`.
- Отключить систему = убрать из списка в `Game`.

## Для углублённого изучения

- [Refactoring Guru: Chain of Responsibility](https://refactoring.guru/design-patterns/chain-of-responsibility)
- Martin R. C. «Clean Architecture», глава о компонентах.

In [ ]:
%%writefile mystic_detective/game/systems.py
"""Игровые системы — обработчики команд игрока.

Каждая система:
- имеет имя (совпадает с Action.name);
- умеет ответить, обрабатывает ли она действие;
- умеет обработать его, изменив GameState.

Реализует принципы:
- SRP: одна ответственность на систему;
- OCP: новая команда = новая система;
- LSP: все системы взаимозаменяемы через GameSystem.
"""

from core.action import Action
from core.game_state import GameState
from ui.ui import UI


class InvestigationSystem:
    """Обрабатывает команду 'investigate' — осмотр локации."""

    @property
    def name(self) -> str:
        return "investigate"

    def can_handle(self, action: Action) -> bool:
        return action.name == self.name

    def handle(self, action: Action, state: GameState, ui: UI) -> None:
        detective = state.detective
        location = state.current_location
        ui.show_message(detective.investigate(location.name))
        clue = location.search()
        if clue is not None:
            detective.add_clue(clue)
            ui.show_message(f"Найдена улика: {clue}")
        else:
            ui.show_message("Здесь больше ничего нет.")


class DialogueSystem:
    """Обрабатывает команду 'talk' — диалог с персонажем."""

    @property
    def name(self) -> str:
        return "talk"

    def can_handle(self, action: Action) -> bool:
        return action.name == self.name

    def handle(self, action: Action, state: GameState, ui: UI) -> None:
        location = state.current_location
        character = location.get_character()
        if character is None:
            ui.show_message("Здесь не с кем говорить.")
            return
        ui.show_message(f"{character} рассказывает что-то важное...")
        div = state.detective.div
        if div is not None:
            ui.show_message(div.speak())


class MovementSystem:
    """Обрабатывает команду 'move' — перемещение между локациями."""

    @property
    def name(self) -> str:
        return "move"

    def can_handle(self, action: Action) -> bool:
        return action.name == self.name

    def handle(self, action: Action, state: GameState, ui: UI) -> None:
        if not action.args:
            ui.show_message("Не указана локация.")
            return
        target = action.args[0]
        if state.move_to(target):
            ui.show_message(f"Вы переместились в {target}.")
            state.detective.rest(5)
        else:
            ui.show_message(f"Локации {target!r} не существует.")


---
# Шаг 7. `VictoryChecker` — проверка победы

## Зачем этот шаг

**Проблема:** условие победы (`len(key_clues) >= 2`) зашито прямо в `Game.run()`. Это:
- нарушает SRP (проверка победы — не дело игрового цикла);
- не позволяет менять условие без правки `Game`;
- не тестируется в изоляции.

**Решение:** вынести условие победы в отдельный класс `VictoryChecker`.

## Теория: политика vs механика

**Политика победы** — это правило игры. Оно может меняться:
- «найти 2 ключевые улики»;
- «найти все улики»;
- «убедить Кузю помочь»;
- «раскрыть правду о Владимире».

Вынесение политики в отдельный класс позволяет:
- менять условие без правки `Game`;
- тестировать разные политики в изоляции;
- комбинировать несколько условий (например, улики + лояльность дива).

## Как это улучшит проект

**До:**
```python
key_clues = [c for c in self._detective.clues if c.is_key]
if len(key_clues) >= 2:
    ...
```

**После:**
```python
if self._victory.is_victory(state):
    ...
```

Условие победы — в одном месте. Легко поменять или комбинировать.

## Для углублённого изучения

- Martin Fowler. «Refactoring», Extract Class.
- [Refactoring Guru: Strategy](https://refactoring.guru/design-patterns/strategy) — паттерн, к которому мы придём в модуле 5.

In [ ]:
%%writefile mystic_detective/game/victory_checker.py
"""VictoryChecker — политика победы.

Ответственность (SRP):
- решать, победил ли игрок, по состоянию GameState.

НЕ отвечает за:
- изменение состояния;
- вывод.

Политику легко заменить: другая VictoryChecker — другое условие.
"""

from core.game_state import GameState


class VictoryChecker:
    """Победа: найдено достаточно ключевых улик."""

    def __init__(self, required_key_clues: int = 2):
        if required_key_clues < 1:
            raise ValueError("required_key_clues должен быть >= 1")
        self._required = required_key_clues

    @property
    def required(self) -> int:
        return self._required

    def is_victory(self, state: GameState) -> bool:
        """Проверить условие победы."""
        key_clues = [c for c in state.detective.clues if c.is_key]
        return len(key_clues) >= self._required


---
# Шаг 8. Новый `Game` — оркестратор через DIP

## Зачем этот шаг

Мы разнесли `Game` на системы. Теперь нужно собрать их обратно в **тонкий оркестратор**, который:
- **не знает** о конкретных системах — только о `GameSystem`;
- **не создаёт** зависимости — получает их через конструктор (DIP);
- **не знает** о UI — системы работают с ним через абстракцию.

## Теория: Dependency Inversion Principle

**DIP** говорит: модули высокого уровня не должны зависеть от модулей низкого уровня. Оба должны зависеть от абстракций.

**До:** `Game` создаёт `Detective`, `Div`, `Location` — он знает о них всё.

**После:** `Game` получает `state`, `systems`, `input_handler`, `renderer`, `victory_checker` через конструктор. Он знает только:
- `GameState` — интерфейс состояния;
- `GameSystem` (Protocol) — интерфейс системы;
- `InputHandler` — интерфейс ввода;
- `Renderer` — интерфейс вывода;
- `VictoryChecker` — интерфейс политики победы.

**Что это даёт:**
- Тесты могут передать mock-систему, mock-состояние, mock-UI.
- Можно заменить `Renderer` на Pygame-версию, не меняя `Game`.
- Можно собрать другую игру из тех же систем.

## Как это улучшит проект

**До:** 200+ строк монолита, 7 ответственностей, невозможно тестировать.

**После:** 40–50 строк оркестрации, каждая часть тестируется отдельно, легко менять поведение.

## Для углублённого изучения

- Martin R. C. «Clean Architecture», глава 11 (DIP).
- [Real Python: Dependency Injection](https://realpython.com/dependency-injection-python/)
- [Martin Fowler: Inversion of Control](https://martinfowler.com/articles/injection.html)

In [ ]:
%%writefile mystic_detective/game/game_v3.py
"""Game — тонкий оркестратор систем.

Принципы:
- SRP: только оркестрация, ничего больше.
- DIP: получает зависимости через конструктор.
- OCP: список систем можно расширять без правки Game.

Сравните с game_before.py — разница в 5 раз по количеству строк.
"""

from core.action import Action
from core.game_state import GameState
from game.input_handler import InputHandler
from game.renderer import Renderer
from game.victory_checker import VictoryChecker
from ui.ui import UI


class Game:
    """Оркестратор: собирает системы и управляет игровым циклом."""

    def __init__(
        self,
        state: GameState,
        systems: list,
        input_handler: InputHandler,
        renderer: Renderer,
        victory_checker: VictoryChecker,
        ui: UI,
    ):
        self._state = state
        self._systems = systems
        self._input = input_handler
        self._renderer = renderer
        self._victory = victory_checker
        self._ui = ui

    def run(self) -> None:
        """Игровой цикл: рендер → ввод → обработка → проверка победы."""
        self._renderer.show_title("ТАЙНА ИСЧЕЗНУВШЕГО ДИВА")

        while self._state.detective.energy > 0:
            self._renderer.render(self._state)
            self._renderer.show_menu()

            action = self._input.read_action()
            if action is None:
                self._ui.show_message("Неизвестная команда.")
                continue

            if action.name == "quit":
                return

            self._dispatch(action)

            if self._victory.is_victory(self._state):
                self._ui.show_message("\nВы раскрыли тайну исчезновения Владимира!")
                return

        self._ui.show_message("\nЭнергия закончилась. Расследование провалено.")

    def _dispatch(self, action: Action) -> None:
        """Найти систему, обрабатывающую действие, и вызвать её."""
        for system in self._systems:
            if system.can_handle(action):
                system.handle(action, self._state, self._ui)
                return
        self._ui.show_message(f"Нет системы для команды {action.name!r}")


## Сборка проекта: фабрика `build_game`

Чтобы клиент не собирал всё вручную, добавим функцию `build_game(ui)`. Это **Composition Root** — единственное место, где создаются конкретные реализации.

In [ ]:
%%writefile mystic_detective/game/factory.py
"""Composition Root: единственное место, где создаются конкретные реализации.

Здесь мы соединяем абстракции с конкретными классами.
Всё остальное в проекте зависит только от абстракций (DIP).
"""

from core.config import DetectiveConfig, GameConfig
from core.game_state import GameState
from entities.clue import Clue
from entities.detective_v2 import Detective
from entities.div import Div
from entities.location import Location
from game.game_v3 import Game
from game.input_handler import InputHandler
from game.renderer import Renderer
from game.systems import DialogueSystem, InvestigationSystem, MovementSystem
from game.victory_checker import VictoryChecker
from ui.ui import UI


def build_locations() -> dict[str, Location]:
    """Создать локации игры."""
    loc1 = Location("особняк", "Старый особняк Авериных на Петроградке.")
    loc1.add_clue(Clue("Странный запах гари", "Пахнет серой", is_key=False))
    loc1.add_character("Дворецкий")

    loc2 = Location("академия", "Академия колдовских наук.")
    loc2.add_clue(Clue("Записка с планом Пустоши", "Карта", is_key=True))
    loc2.add_character("Ректор")

    loc3 = Location("пустошь", "Ледяная пустыня, мир дивов.")
    loc3.add_clue(Clue("Следы Владимира", "Отпечатки лап", is_key=True))
    loc3.add_character("Древний див")

    return {"особняк": loc1, "академия": loc2, "пустошь": loc3}


def build_game(ui: UI | None = None) -> Game:
    """Собрать игру: создать состояние, системы и оркестратор."""
    ui = ui if ui is not None else UI()

    div = Div("Кузя", form="кот", loyalty=50)
    detective = Detective(
        config=DetectiveConfig(),
        game_config=GameConfig(),
        div=div,
    )
    locations = build_locations()
    state = GameState(detective, locations, start_location="особняк")

    systems = [
        InvestigationSystem(),
        DialogueSystem(),
        MovementSystem(),
    ]

    return Game(
        state=state,
        systems=systems,
        input_handler=InputHandler(ui),
        renderer=Renderer(ui),
        victory_checker=VictoryChecker(required_key_clues=2),
        ui=ui,
    )


### Демонстрация: игра работает как раньше

In [ ]:
import sys
sys.path.insert(0, 'mystic_detective')

from ui.ui import FakeUI
from game.factory import build_game

fake = FakeUI([
    '1',        # осмотреть особняк
    '3',        # переместиться
    'академия', # в академию
    '1',        # осмотреть — ключевая улика
    '3',        # переместиться
    'пустошь',  # в пустошь
    '1',        # осмотреть — вторая ключевая улика
])

game = build_game(ui=fake)
game.run()

---
# Шаг 9. Демонстрация OCP: новая команда без правки `Game`

## Зачем этот шаг

Это доказательство, что OCP работает. Мы добавим новую команду «Применить Плеть» — и покажем, что **`Game` не меняется**.

## Что делаем

1. Создаём `WhipSystem` в новом файле.
2. Добавляем команду `'5'` в `InputHandler.COMMANDS` (единственная правка ввода).
3. Добавляем систему в список `systems` в `build_game`.

**`Game.run()` не трогаем.** Это и есть OCP.

## Теория: почему это работает

`Game.run()` вызывает `self._dispatch(action)`, который ищет систему через `can_handle`. Новая система автоматически подхватывается — `Game` не знает о ней конкретно.

## Для углублённого изучения

- [Refactoring Guru: Open/Closed Principle](https://refactoring.guru/design-patterns/open-closed-principle)

In [ ]:
%%writefile mystic_detective/game/whip_system.py
"""WhipSystem — новая команда «Применить Плеть».

Демонстрирует OCP: Game не знает о ней конкретно,
но она автоматически работает через can_handle/handle.
"""

from core.action import Action
from core.game_state import GameState
from ui.ui import UI


class WhipSystem:
    """Обрабатывает команду 'whip' — применение Плети."""

    @property
    def name(self) -> str:
        return "whip"

    def can_handle(self, action: Action) -> bool:
        return action.name == self.name

    def handle(self, action: Action, state: GameState, ui: UI) -> None:
        result = state.detective.use_whip()
        ui.show_message(result)


In [ ]:
# Расширим InputHandler новым кодом '5' — единственная правка ввода.
# (в реальном проекте это делается в самом классе, здесь для демонстрации — monkeypatch)

from game.input_handler import InputHandler

InputHandler.COMMANDS['5'] = 'whip'
print("Команды ввода:", InputHandler.COMMANDS)

In [ ]:
# Собираем игру заново, добавив WhipSystem в список систем
from game.factory import build_game
from game.whip_system import WhipSystem
from ui.ui import FakeUI

fake = FakeUI(['5', '5', '4'])  # whip, whip, quit
game = build_game(ui=fake)
game._systems.append(WhipSystem())
game.run()

print("\n✅ Новая команда работает без правки Game!")

---
# Шаг 10. Юнит-тесты

## Зачем этот шаг

Мы провели крупный рефакторинг. **Главный риск:** сломать существующее поведение. Тесты из заданий №1 и №2 должны продолжать проходить. Плюс добавим тесты на новые классы.

## Теория: что тестировать

**Принципы тестирования после рефакторинга:**
1. **Существующие тесты должны проходить.** Если что-то сломалось — это регрессия.
2. **Новые классы — новые тесты.** `Action`, `GameState`, `InputHandler`, `Renderer`, системы, `VictoryChecker`.
3. **Мок-объекты для изоляции.** Системы тестируются с mock-UI, `GameState` — с mock-детективом (или с реальным).

## Что тестируем

| Что | Как |
|---|---|
| `Action` | frozen, равенство, `__str__` |
| `GameState` | move_to, turn, current_location |
| `InputHandler` | команды, неизвестный ввод |
| `VictoryChecker` | условия победы |
| Системы | can_handle, handle с FakeUI |
| Регрессия | старые тесты из заданий №1, №2 |

## Для углублённого изучения

- [pytest documentation](https://docs.pytest.org/)
- [Real Python: Testing Your Code with pytest](https://realpython.com/pytest-python-testing/)
- Martin Fowler. «Refactoring», глава о тестировании.

In [ ]:
%%writefile mystic_detective/tests/test_action.py
"""Тесты для Action."""

import pytest
from core.action import Action


class TestAction:
    def test_create_simple_action(self):
        a = Action("investigate")
        assert a.name == "investigate"
        assert a.args == ()

    def test_create_action_with_args(self):
        a = Action("move", ("академия",))
        assert a.args == ("академия",)

    def test_action_is_frozen(self):
        a = Action("x")
        with pytest.raises(Exception):
            a.name = "y"

    def test_action_equality(self):
        assert Action("x") == Action("x")
        assert Action("x") != Action("y")

    def test_action_str(self):
        assert str(Action("x")) == "x"
        assert str(Action("move", ("a",))) == "move(a)"


In [ ]:
%%writefile mystic_detective/tests/test_game_state.py
"""Тесты для GameState."""

import pytest
from core.game_state import GameState
from entities.location import Location


class MockDetective:
    pass


class TestGameState:
    def _make_state(self):
        locs = {
            "a": Location("a"),
            "b": Location("b"),
        }
        return GameState(MockDetective(), locs, start_location="a")

    def test_initial_location(self):
        s = self._make_state()
        assert s.current_location_name == "a"

    def test_move_to_valid(self):
        s = self._make_state()
        assert s.move_to("b")
        assert s.current_location_name == "b"

    def test_move_to_invalid(self):
        s = self._make_state()
        assert not s.move_to("c")
        assert s.current_location_name == "a"

    def test_turn_increments_on_move(self):
        s = self._make_state()
        assert s.turn == 0
        s.move_to("b")
        assert s.turn == 1

    def test_invalid_start_location(self):
        with pytest.raises(ValueError):
            GameState(MockDetective(), {}, start_location="x")


In [ ]:
%%writefile mystic_detective/tests/test_victory_checker.py
"""Тесты для VictoryChecker."""

import pytest
from core.game_state import GameState
from entities.clue import Clue
from entities.location import Location
from game.victory_checker import VictoryChecker


class MockDetectiveWithClues:
    def __init__(self, clues):
        self._clues = clues

    @property
    def clues(self):
        return self._clues


def make_state_with_clues(clues):
    locs = {"x": Location("x")}
    return GameState(MockDetectiveWithClues(clues), locs, "x")


class TestVictoryChecker:
    def test_not_victory_initially(self):
        checker = VictoryChecker(required_key_clues=2)
        state = make_state_with_clues([])
        assert not checker.is_victory(state)

    def test_victory_with_enough_key_clues(self):
        checker = VictoryChecker(required_key_clues=2)
        state = make_state_with_clues([
            Clue("a", is_key=True),
            Clue("b", is_key=True),
        ])
        assert checker.is_victory(state)

    def test_non_key_clues_dont_count(self):
        checker = VictoryChecker(required_key_clues=2)
        state = make_state_with_clues([
            Clue("a", is_key=True),
            Clue("b", is_key=False),
        ])
        assert not checker.is_victory(state)

    def test_invalid_required(self):
        with pytest.raises(ValueError):
            VictoryChecker(required_key_clues=0)


In [ ]:
%%writefile mystic_detective/tests/test_systems.py
"""Тесты для игровых систем."""

from core.action import Action
from core.game_state import GameState
from entities.clue import Clue
from entities.detective_v2 import Detective
from entities.location import Location
from game.systems import DialogueSystem, InvestigationSystem, MovementSystem
from ui.ui import FakeUI


def make_state():
    det = Detective()
    loc = Location("особняк", "Старый особняк")
    loc.add_clue(Clue("Запах гари", is_key=False))
    loc.add_character("Дворецкий")
    locs = {"особняк": loc, "академия": Location("академия")}
    return GameState(det, locs, "особняк")


class TestInvestigationSystem:
    def test_can_handle(self):
        s = InvestigationSystem()
        assert s.can_handle(Action("investigate"))
        assert not s.can_handle(Action("talk"))

    def test_handle_finds_clue(self):
        state = make_state()
        ui = FakeUI([])
        InvestigationSystem().handle(Action("investigate"), state, ui)
        assert len(state.detective.clues) == 1


class TestDialogueSystem:
    def test_handle_with_character(self):
        state = make_state()
        ui = FakeUI([])
        DialogueSystem().handle(Action("talk"), state, ui)
        assert any("Дворецкий" in m for m in ui.messages)


class TestMovementSystem:
    def test_move_valid(self):
        state = make_state()
        ui = FakeUI([])
        MovementSystem().handle(Action("move", ("академия",)), state, ui)
        assert state.current_location_name == "академия"

    def test_move_invalid(self):
        state = make_state()
        ui = FakeUI([])
        MovementSystem().handle(Action("move", ("пустошь",)), state, ui)
        assert state.current_location_name == "особняк"


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-v', '--tb=short'],
    cwd='mystic_detective',
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Код выхода: {result.returncode}")

---
# Шаг 11. Документация

## Зачем этот шаг

Рефакторинг по SOLID — крупное архитектурное изменение. Без документации через месяц никто (включая вас) не поймёт, почему `Game` стал таким тонким.

## Что документируем

1. **README** — обновить раздел «Архитектура»: новые слои (`core`, `game`, `entities`, `ui`).
2. **ADR-003** — решение о разбиении `Game` на системы. Контекст, альтернативы, последствия.
3. **Диаграмма классов** — обновить UML с учётом новых классов.

## Теория: ADR для рефакторинга

ADR особенно важен для рефакторинга, потому что:
- фиксирует **проблему** («God Object»);
- описывает **решение** (системы + DIP);
- перечисляет **альтернативы** (например, «оставить как есть», «использовать события вместо систем»);
- объясняет **последствия** (плюсы: расширяемость; минусы: косвенность).

## Для углублённого изучения

- [Michael Nygard: Documenting Architecture Decisions](https://cognitect.com/blog/2011/11/15/documenting-architecture-decisions)

In [ ]:
%%writefile mystic_detective/docs/adr_003_solid.md
# ADR-003: Разбиение Game на системы по SOLID

## Контекст
Класс Game после заданий №1–№2 стал «божественным объектом»:
- 7 ответственностей в одном классе (SRP);
- if/elif по командам (OCP);
- создание Detective, Div, Location внутри конструктора (DIP);
- невозможно тестировать игровой цикл изолированно.

## Решение
Разбить Game на тонкий оркестратор и набор систем:
- Action — типизированный объект команды (dataclass);
- GameState — состояние игры;
- GameSystem (Protocol) — контракт для обработчиков;
- InputHandler, Renderer — ввод и вывод;
- InvestigationSystem, DialogueSystem, MovementSystem — обработчики команд;
- VictoryChecker — политика победы;
- Game — оркестратор, получающий все зависимости через конструктор.

## Рассмотренные альтернативы
1. **Оставить God Object.** Минус: регрессии при каждом изменении.
2. **Использовать события (EventBus).** Преждевременно: паттерн Observer — тема модуля 6.
3. **Разбить на системы через Protocol (выбрано).** Плюсы: расширяемость, тестируемость.

## Последствия
- ✅ Добавление новой команды = новый класс, без правки Game (OCP).
- ✅ Каждая система тестируется изолированно.
- ✅ Renderer можно заменить на Pygame-версию без правки логики.
- ⚠️ Больше классов, выше косвенность.
- ⚠️ Новичку сложнее проследить путь команды (input → dispatch → system).


In [ ]:
%%writefile mystic_detective/docs/uml_module3.md
```mermaid
classDiagram
    class Game {
        -_state: GameState
        -_systems: list[GameSystem]
        -_input: InputHandler
        -_renderer: Renderer
        -_victory: VictoryChecker
        +run() void
        -_dispatch(action) void
    }
    class GameState {
        -_detective: Detective
        -_locations: dict
        -_current_location_name: str
        +move_to(name) bool
        +current_location Location
    }
    class InputHandler {
        +read_action() Action
    }
    class Renderer {
        +render(state) void
        +show_menu() void
    }
    class VictoryChecker {
        +is_victory(state) bool
    }
    class GameSystem {
        <<Protocol>>
        +name str
        +can_handle(action) bool
        +handle(action, state, ui) void
    }
    class InvestigationSystem {
        +can_handle() bool
        +handle() void
    }
    class DialogueSystem {
        +can_handle() bool
        +handle() void
    }
    class MovementSystem {
        +can_handle() bool
        +handle() void
    }
    Game --> GameState : holds
    Game --> InputHandler : uses
    Game --> Renderer : uses
    Game --> VictoryChecker : uses
    Game --> GameSystem : dispatches
    GameSystem <|.. InvestigationSystem
    GameSystem <|.. DialogueSystem
    GameSystem <|.. MovementSystem
```


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pycodestyle',
     'core/action.py', 'core/game_system.py', 'core/game_state.py',
     'game/input_handler.py', 'game/renderer.py',
     'game/systems.py', 'game/victory_checker.py', 'game/game_v3.py'],
    cwd='mystic_detective',
    capture_output=True,
    text=True,
)
if result.stdout:
    print("⚠️ Замечания PEP 8:")
    print(result.stdout)
else:
    print("✅ PEP 8: замечаний нет")

---
# Итоги: что мы сделали и как это улучшило проект

## До и после

| Аспект | До (God Object) | После (системы + DIP) |
|---|---|---|
| Ответственностей в `Game` | 7 | 1 (оркестрация) |
| Строк в `Game` | ~90 | ~50 |
| Добавить команду | Правка `Game.run()` | Новый класс-система |
| Заменить UI | Правка `Game` | Передать другой `UI` в фабрику |
| Заменить условие победы | Правка `Game` | Передать другой `VictoryChecker` |
| Тестировать команду | Невозможно | `InvestigationSystem().handle(...)` |
| Заменить состояние | Невозможно | Передать mock-`GameState` |

## Что улучшилось по принципам

- **SRP:** каждый класс делает одно.
- **OCP:** новая команда = новая система, `Game` не меняется.
- **LSP:** все системы взаимозаменяемы через `GameSystem`.
- **ISP:** `GameSystem` требует только `name`, `can_handle`, `handle` — минимум.
- **DIP:** `Game` зависит от абстракций, конкретика собирается в `build_game`.

## Что дальше

В модуле 4 мы добавим паттерн **Game Loop** с фиксированным шагом и паттерн **Component**:
- `update(dt)` станет ключевым методом;
- компоненты (`Position`, `Health`, `Velocity`) выделятся в dataclass;
- игровой цикл начнёт работать с миллисекундами.

## Литература

**Базовая:**
- Martin R. C. «Clean Architecture», часть III (SOLID).
- Martin R. C. «Clean Code», главы 3, 10.
- Лутц М. «Изучаем Python», главы 26–27.

**Углублённая:**
- Martin Fowler. «Refactoring: Improving the Design of Existing Code».
- [Refactoring Guru: SOLID](https://refactoring.guru/design-patterns/solid)
- [Real Python: SOLID Principles in Python](https://realpython.com/solid-principles-python/)
- [Real Python: Dependency Injection](https://realpython.com/dependency-injection-python/)
- [Martin Fowler: Inversion of Control](https://martinfowler.com/articles/injection.html)
- [Python docs: typing.Protocol](https://typing.python.org/en/latest/spec/protocol.html)
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)